# 12. スキーマ進化 - 列が増えたときに何が起きるか

`01` で Auto Loader を使ったとき、スキーマを自分で書きませんでした。
JSONを読んで、Databricksが勝手に決めてくれていました。

便利ですが、ひとつ問題があります。**上流はこちらの都合に関係なく変わります。**

- 新しい列が増えた
- 今まで数値だった列に文字列が入ってきた
- 列が無くなった

こういうとき、取り込みは止まるのか、黙って進むのか。
黙って進むなら、そのデータはどこへ行くのか。ここを確かめます。

このノートブックで確かめること:

1. 列が増えると、既定ではどうなるか
2. なぜそういう設計なのか
3. 止めずに流す方法
4. モードをどう選ぶか

**前提**: `00_setup` を実行済みであること。`01` を読んでいること。


## 準備


In [1]:
import json

from databricks.connect import DatabricksSession
from databricks.sdk.errors import NotFound
from databricks.sdk.runtime import dbutils

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [2]:
CATALOG = "tech_survey"

# 既定モードを見る側
TABLE = f"{CATALOG}.bronze.schema_default"
LANDING = f"/Volumes/{CATALOG}/ops/landing/12_schema_default"
CHECKPOINT = f"/Volumes/{CATALOG}/ops/checkpoints/12_schema_default"

# rescue モードを見る側。3章で使う
TABLE_RESCUE = f"{CATALOG}.bronze.schema_rescue"
LANDING_RESCUE = f"/Volumes/{CATALOG}/ops/landing/12_schema_rescue"
CHECKPOINT_RESCUE = f"/Volumes/{CATALOG}/ops/checkpoints/12_schema_rescue"

In [ ]:
def run_stream(landing: str, checkpoint: str, table: str, mode: str | None = None) -> None:
    """
    Auto Loader で landing を読み、table へ追記する。

    Parameters
    ----------
    landing : str
        取り込み元のフォルダ。
    checkpoint : str
        チェックポイントの置き場所。推論したスキーマもこの下に記録される。
    table : str
        書き込み先のテーブル。
    mode : str or None, default None
        `cloudFiles.schemaEvolutionMode` に渡す値。None なら既定のまま。

    Notes
    -----
    書き込み側に付けている `mergeSchema` については2章で扱う。
    """
    reader = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{checkpoint}/_schema")
    )
    if mode:
        reader = reader.option("cloudFiles.schemaEvolutionMode", mode)

    query = (
        reader.load(landing)
        .writeStream.option("checkpointLocation", checkpoint)
        .option("mergeSchema", "true")  # NOTE: mergeSchema は Deltaの安全装置を緩める指定
        .trigger(availableNow=True)
        .toTable(table)
    )
    query.awaitTermination()

## 1. まず普通に取り込む

2列 (`order_id`, `product`) のJSONを1件置いて、取り込みます。


In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
for path in (LANDING, CHECKPOINT):
    try:
        dbutils.fs.rm(path, True)
    except NotFound:
        pass

dbutils.fs.put(f"{LANDING}/f1.json", json.dumps({"order_id": 1, "product": "laptop"}), True)

run_stream(LANDING, CHECKPOINT, TABLE, mode=None)

display(spark.table(TABLE))

,order_id,product,_rescued_data
0,1,laptop,None


列を見てください。`order_id` と `product` のほかに **`_rescued_data`** が付いています。

これはこちらが作った列ではありません。**Auto Loaderが自動で足します。**
「スキーマに収まらなかったものを入れる箱」で、いまは空です。3章で中身が入ります。

推論したスキーマは `cloudFiles.schemaLocation` に指定した場所へ記録されています。
`01` でチェックポイントを消したら取り込み直しになったのは、ここも一緒に消えていたからです。

**Auto Loaderにとってスキーマは「覚えておくもの」** で、毎回ファイルから決め直しているわけではありません。


## 2. 列が増えると落ちる

上流が `note` という列を足してきた、という想定でファイルを1つ置きます。
そのまま同じストリームを実行します。何が起きるか予想してください。


In [5]:
dbutils.fs.put(
    f"{LANDING}/f2.json",
    json.dumps({"order_id": 2, "product": "monitor", "note": "急ぎ"}),
    True,
)

# エラーメッセージを読みたいので、例外を捕まえて表示する
try:
    run_stream(LANDING, CHECKPOINT, TABLE)
    print("通った")
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:300])

StreamingQueryException
[STREAM_FAILED] Query [id = e1d4a569-0454-4544-b295-f839d0d65bab, runId = 5bed021e-487d-4cbe-ba66-f713a8a9a9ed] terminated with exception: [UNKNOWN_FIELD_EXCEPTION.NEW_FIELDS_IN_RECORD_WITH_FILE_PATH] Encountered unknown fields during parsing: {"note":"急ぎ"}, which can be fixed by an automatic retry:


**ストリームが落ちます。** `UNKNOWN_FIELD_EXCEPTION` です。

一見すると不便です。列が増えただけで処理が止まるのですから。
ですがこれは、既定の `addNewColumns` モードの **設計どおり** の動きです。

落ちる前に、Auto Loaderは **新しい列を含めたスキーマを `schemaLocation` に書き直しています。**
つまりこの失敗は「知らない列が来たので覚え直した。次からは扱える」という通知です。

そのまま、もう一度実行します。


In [6]:
run_stream(LANDING, CHECKPOINT, TABLE)

display(spark.table(TABLE))

,order_id,product,_rescued_data,note
0,2,monitor,None,急ぎ
1,1,laptop,None,None


`note` が列として増え、2件とも入っています。

**落ちたときのデータは失われていません。**
処理が完了していないのでチェックポイントを進めておらず、再実行で両方とも取り込まれます。
`06` の「同じ処理をやり直しても重複しない」と同じ考え方です。

ところで、`run_stream` の書き込み側に付けている `mergeSchema` は何をしているのか。
ここが分かっていないと、2章で起きたことの半分しか見えていないことになります。


### `mergeSchema` とは

Deltaテーブルは既定で **スキーマを強制します。**
書き込もうとしているデータの形がテーブルと違うと、**書き込みを拒否します。**

```
[DELTA_METADATA_MISMATCH] A schema mismatch detected when writing to the Delta table
```

これは安全装置です。
上流のバグや書き間違いで変な列が紛れ込んだとき、
黙ってテーブルの形が変わっていくのを防ぎます。

`09` の expectations が **行の中身** を守るものだとすれば、
こちらは **テーブルの形** を守るものです。

`.option("mergeSchema", "true")` は、その安全装置を
**「列が増えるぶんには広げてよい」** に緩める指定です。

### 関門は2つある

つまりこのノートブックでは、**別々の2つの関門** を通っています。

| | どこの話か | 何を見ているか | 指定 |
|---|---|---|---|
| 読み取り側 | Auto Loader | ファイルの中に、覚えているスキーマに無い列があるか | `cloudFiles.schemaEvolutionMode` |
| 書き込み側 | Deltaテーブル | 書き込むデータに、テーブルに無い列があるか | `mergeSchema` |

2章で起きたことを順に追うと、こうなります。

1. `note` を含むファイルが届く
2. **読み取り側** が「知らない列だ」と気づいて落ちる。同時にスキーマを覚え直す
3. 再実行。今度は読み取り側を通り、`note` を持つデータができる
4. **書き込み側** に届く。テーブルにはまだ `note` が無い
5. `mergeSchema` があるので、テーブルに `note` を足して書き込む

`mergeSchema` を外すと、3までは同じで **5で失敗します。**
「読み取り側の問題は解決したのに、まだ落ちる」という状態になり、原因が分かりにくくなります。

なお `mergeSchema` が面倒を見るのは **列が増える方向だけ** です。
列を消したり、型を非互換に変えたりはできません。


## 3. 落とさずに流す - `rescue` モード

「列が増えるたびにジョブが1回落ちる」のは、運用によっては困ります。
夜間バッチが落ちて朝に気づく、という形になるからです。

`cloudFiles.schemaEvolutionMode` を `rescue` にすると **落ちずに流し続けます。**
新しい列はテーブルに足さず、`_rescued_data` にしまい込みます。

別のテーブルで試します。今度は `order_id` と `amount` の2列から始めて、`extra` を足します。


In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE_RESCUE}")
for path in (LANDING_RESCUE, CHECKPOINT_RESCUE):
    try:
        dbutils.fs.rm(path, True)
    except NotFound:
        pass

dbutils.fs.put(f"{LANDING_RESCUE}/f1.json", json.dumps({"order_id": 1, "amount": 100}), True)
run_stream(LANDING_RESCUE, CHECKPOINT_RESCUE, TABLE_RESCUE, mode="rescue")

# 新しい列 extra を含むファイルを足して、もう一度流す
dbutils.fs.put(
    f"{LANDING_RESCUE}/f2.json",
    json.dumps({"order_id": 2, "amount": 200, "extra": "x"}),
    True,
)
run_stream(LANDING_RESCUE, CHECKPOINT_RESCUE, TABLE_RESCUE, mode="rescue")  # ここが rescue モード

display(spark.table(TABLE_RESCUE).orderBy("order_id"))

,amount,order_id,_rescued_data
0,100,1,None
1,200,2,"{""extra"":""x"",""_file_path"":""/Volumes/tech_survey/ops/landing/12_schema_rescue/f2.json""}"


ストリームは落ちませんでした。そして `extra` は **列として増えていません。**

代わりに `_rescued_data` にJSONで入っています。

```json
{"extra":"x","_file_path":"/Volumes/.../12_schema_rescue/f2.json"}
```

`_file_path` も一緒に入るのがポイントです。
**どのファイルから来たのかを後から追える** ので、上流に問い合わせるときの材料になります。

止まらず、捨てもしない。ただし **そのままでは使えない形** で入っています。
使うには後から取り出す必要があります。


### 型に注目

`amount` の値を見てください。JSONでは数値で書いたのに、**文字列として入っています。**

Auto Loaderは既定で、すべての列を **文字列として推論します**
(`cloudFiles.inferColumnTypes` が `false`)。

型を決めないので、「数値のはずの列に文字列が来た」という食い違いがそもそも起きません。
上流が急に `"未定"` のような値を入れてきても、取り込みは止まりません。

`07` で `event_time` を、`08` で `amount` をキャストしていたのはこれが理由です。
**取り込みでは型を決めず、使う側の層で型を付ける** のが既定の流れになっています。
`04` で見たメダリオンの考え方 (Bronzeは生のまま受ける) とも一致します。

`cloudFiles.inferColumnTypes` を `true` にすると型を推論しますが、
今度は型の合わない値が `_rescued_data` に落ちるようになります。
**どちらの箱に入るかが変わるだけで、「捨てない」という方針は同じ** です。


## 4. モードの選び方

`cloudFiles.schemaEvolutionMode` には4つあります。

| モード | 新しい列が来たら | ストリーム |
|---|---|---|
| `addNewColumns` (既定) | 列として足す | **1回落ちる**。再実行で通る |
| `rescue` | `_rescued_data` に入れる | 止まらない |
| `failOnNewColumns` | 何もしない | 落ちる。スキーマを手で直すまで通らない |
| `none` | 無視して捨てる | 止まらない |

選び方はこうなります。

- **`addNewColumns`** … 迷ったらこれ。列は自動で増え、落ちても再実行で回復する。
  ただし **再実行が自動で走る** ようにしておく必要がある。ジョブのリトライ設定で吸収するのが普通 (`13` で扱う)
- **`rescue`** … 止められない取り込みに使う。
  代わりに `_rescued_data` を誰かが見る運用が要る。放っておくと気づかないまま溜まる
- **`failOnNewColumns`** … 列が増えること自体を事故として扱いたいとき。
  スキーマを人が管理している場合に選ぶ
- **`none`** … 捨ててよいと言い切れるときだけ。**後から取り返せない**

どのモードでも共通するのは、**上流の変更に気づく仕組みが別に要る** ことです。
`addNewColumns` は落ちるので気づけますが、`rescue` と `none` は黙って進みます。
`rescue` なら「`_rescued_data` が空でない行を数える」といった監視を用意することになります。

`06` の `txnVersion`、`07` のウォーターマーク、`09` の expectations と同じ形です。
**黙って進むものは、こちらから見に行かないと分かりません。**


## 考えてみる

- `_rescued_data` に入った `extra` を、後から普通の列として使うにはどうしますか
- 上流が列を **削除** したら、どうなるでしょうか
- `addNewColumns` で落ちたとき、ジョブのリトライ任せにしてよいでしょうか


### 答え

**Q1. `_rescued_data` から取り出す**

中身はJSONの文字列なので、`09` のイベントログと同じやり方で取り出せます。

```sql
SELECT _rescued_data:extra AS extra FROM テーブル
```

ただしこれは **その場しのぎ** です。
`extra` を継続的に使うなら、列として取り込めるようにするほうが素直になります。
モードを `addNewColumns` に変えて流し直すか、`schemaLocation` を消して推論からやり直すことになります。

**Q2. 列が削除されたら**

**何も起きません。エラーにもなりません。**

スキーマには列が残ったままで、その列を持たないファイルを読むと **`null` が入ります。**
増えるほうは検知しますが、**減るほうは検知しません。**

「値が来なくなったこと」に気づくには、別の監視が要ります。
`09` の expectations で `列 IS NOT NULL` を見ておく、といった形になります。

**Q3. リトライ任せにしてよいか**

**基本は任せてよい** です。落ちた時点でスキーマは更新済みなので、次の実行で通ります。

ただし **無限にリトライさせてはいけません。**
壊れたファイルが原因で落ちている場合は、何度やっても通らないからです。
回数を決めて、それでも駄目なら通知する形にします。

「1回落ちるが2回目で通る」という前提でリトライを組むと、
**本当に直らない失敗が埋もれます。** リトライ回数と通知はセットで考える必要があります。


## 後片付け

このノートブックで作ったものを消したいときだけ、コメントを外して実行します。


In [8]:
# for t in (TABLE, TABLE_RESCUE):
#     spark.sql(f"DROP TABLE IF EXISTS {t}")
# for path in (LANDING, CHECKPOINT, LANDING_RESCUE, CHECKPOINT_RESCUE):
#     try:
#         dbutils.fs.rm(path, True)
#     except NotFound:
#         pass